In [1]:
from langchain_community.document_loaders import TextLoader,WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
website_url = "https://docs.langchain.com/langsmith/administration-overview"
web_loader = WebBaseLoader(website_url)
web_loader.load()

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusOn this pageResource hierarchyOrganizationsWorkspacesApplicationsResourcesAdditional infoResource tagsUser management and RBACUsersAPI keysExpiration datesPersonal access tokens (PATs)Service keysOrganizati

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [5]:
splitted_docs = text_splitter.split_documents(web_loader.load())

In [6]:
splitted_docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional'),
 Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementAccess control & AuthenticationScalability & 

## embedding with huggingface

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
hugging_embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [8]:
vector_db = FAISS.from_documents(splitted_docs,hugging_embedding)

In [9]:
vector_db.save_local(index_name='faiss_index',folder_path='faiss')

In [11]:
query = 'langsmith is good'
vector_db.similarity_search(query)

[Document(id='c2bfc9eb-37dd-405b-b715-a82061d40524', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='LangSmith resource tags are very similar to tags in cloud services like AWS.'),
 Document(id='5f5b2066-468a-48d8-a7af-7809d4e740af', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='\u200bBilling model\nBillable metrics\nOn your LangSmith invoice, you will see two metrics that we charge for:\n\nLangSmith Traces (Base Charge)\nLangSmith Traces (Extended Data Retention Upgrades).'),
 Document(id='96039260-0507-47e8-bb4d-48f11dd36730', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Tutorial on how to optimize spend\n\n\u200bAdditional resources\n\nRelease Versions: 

### LLM

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_groq import ChatGroq
from langchain_classic.chains.retrieval import create_retrieval_chain

from dotenv import load_dotenv
import os
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_KEY')
llm = ChatGroq(model='openai/gpt-oss-20b',api_key=GROQ_API_KEY)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002096199D190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002096199D970>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [26]:
prompt = ChatPromptTemplate.from_template(
    """
    here is the context for the prompt:
    <context>
     {context}
    </context>
    """
)
prompt

ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    here is the context for the prompt:\n    <context>\n     {context}\n    </context>\n    '), additional_kwargs={})])

In [29]:
docs_chain  = create_stuff_documents_chain(llm,prompt)
docs_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    here is the context for the prompt:\n    <context>\n     {context}\n    </context>\n    '), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020961994920>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020961997320>, model_name='openai/gpt-oss-20b

In [36]:
from langchain_core.documents import Document
docs_chain.invoke({'input':query,'context':[Document(page_content=query)]})


'Thanks for sharing that! How can I help you with LangSmith? If you have a question, need a recommendation, or want to dive into a specific feature, just let me know!'

In [37]:
db_retriever = vector_db.as_retriever()
db_retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000208E95E7F50>, search_kwargs={})

In [38]:
retriever_chain  = create_retrieval_chain(db_retriever,docs_chain)
retriever_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000208E95E7F50>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    here is the context for the prompt:\n    <context>\n     {context}\n    </context>\n    '), additional_kwargs={})])
            | ChatGroq

In [42]:
response = retriever_chain.invoke({
    'input':"summary, implementation guidance for langchain",
})

In [48]:
with open('../artifacts/simple_llm.txt', 'w+',encoding='utf') as f:
    f.write(response['answer'])